# Model Comparison Notebook

Compare different model architectures, precisions, and optimizations for retail vision analytics.

## Contents
1. Model Loading & Setup
2. Accuracy Comparison
3. Latency Benchmarks
4. Memory Analysis
5. Precision vs Performance Trade-offs
6. Recommendations

In [ ]:
# Standard imports
import sys
import time
import json
from pathlib import Path
from typing import Dict, List, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Add project root
sys.path.insert(0, str(Path.cwd().parent))

print("Environment ready!")

## 1. Model Configuration

In [ ]:
# Define models to compare
MODELS = {
    'YOLOv8n-FP32': {
        'path': 'data/models/yolov8n_retail_fp32.engine',
        'precision': 'fp32',
        'size': 'nano',
    },
    'YOLOv8n-FP16': {
        'path': 'data/models/yolov8n_retail_fp16.engine',
        'precision': 'fp16',
        'size': 'nano',
    },
    'YOLOv8n-INT8': {
        'path': 'data/models/yolov8n_retail_int8.engine',
        'precision': 'int8',
        'size': 'nano',
    },
    'YOLOv8s-FP16': {
        'path': 'data/models/yolov8s_retail_fp16.engine',
        'precision': 'fp16',
        'size': 'small',
    },
    'YOLOv8m-FP16': {
        'path': 'data/models/yolov8m_retail_fp16.engine',
        'precision': 'fp16',
        'size': 'medium',
    },
}

# Test configuration
INPUT_SHAPE = (640, 640)
BATCH_SIZE = 1
WARMUP_ITERATIONS = 50
BENCHMARK_ITERATIONS = 500

print(f"Models to compare: {len(MODELS)}")
for name, config in MODELS.items():
    print(f"  - {name}: {config['precision']} precision, {config['size']} size")

## 2. Benchmark Functions

In [ ]:
def mock_benchmark(model_name: str, iterations: int = 100) -> Dict[str, float]:
    """
    Mock benchmark for demonstration.
    In production, replace with actual TensorRT inference.
    """
    # Simulate different performance based on model
    base_latencies = {
        'YOLOv8n-FP32': (8, 12),
        'YOLOv8n-FP16': (4, 6),
        'YOLOv8n-INT8': (2, 4),
        'YOLOv8s-FP16': (8, 12),
        'YOLOv8m-FP16': (15, 22),
    }
    
    low, high = base_latencies.get(model_name, (5, 10))
    latencies = np.random.uniform(low, high, iterations)
    
    return {
        'model': model_name,
        'iterations': iterations,
        'mean_ms': np.mean(latencies),
        'std_ms': np.std(latencies),
        'min_ms': np.min(latencies),
        'max_ms': np.max(latencies),
        'p50_ms': np.percentile(latencies, 50),
        'p95_ms': np.percentile(latencies, 95),
        'p99_ms': np.percentile(latencies, 99),
        'fps': 1000 / np.mean(latencies),
        'latencies': latencies,
    }

def mock_accuracy(model_name: str) -> Dict[str, float]:
    """
    Mock accuracy metrics for demonstration.
    """
    # Simulate accuracy differences
    base_map = {
        'YOLOv8n-FP32': 0.72,
        'YOLOv8n-FP16': 0.72,  # FP16 typically matches FP32
        'YOLOv8n-INT8': 0.70,  # Slight accuracy loss
        'YOLOv8s-FP16': 0.78,
        'YOLOv8m-FP16': 0.82,
    }
    
    mAP = base_map.get(model_name, 0.70) + np.random.uniform(-0.01, 0.01)
    
    return {
        'model': model_name,
        'mAP@0.5': mAP,
        'mAP@0.5:0.95': mAP - 0.15,
        'precision': mAP + 0.05,
        'recall': mAP - 0.02,
    }

print("Benchmark functions defined.")

## 3. Run Benchmarks

In [ ]:
# Run benchmarks for all models
benchmark_results = []
accuracy_results = []

print("Running benchmarks...")
for model_name in MODELS:
    print(f"  Benchmarking {model_name}...")
    
    # Latency benchmark
    bench = mock_benchmark(model_name, BENCHMARK_ITERATIONS)
    benchmark_results.append(bench)
    
    # Accuracy evaluation
    acc = mock_accuracy(model_name)
    accuracy_results.append(acc)

print("\nBenchmarks complete!")

# Create DataFrames
bench_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'latencies'} 
                         for r in benchmark_results])
acc_df = pd.DataFrame(accuracy_results)

bench_df

## 4. Latency Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Mean latency bar chart
ax1 = axes[0]
colors = sns.color_palette('husl', len(MODELS))
bars = ax1.bar(bench_df['model'], bench_df['mean_ms'], color=colors, edgecolor='black')
ax1.errorbar(bench_df['model'], bench_df['mean_ms'], yerr=bench_df['std_ms'], 
             fmt='none', color='black', capsize=5)
ax1.set_xlabel('Model')
ax1.set_ylabel('Latency (ms)')
ax1.set_title('Mean Inference Latency')
ax1.tick_params(axis='x', rotation=45)

# Add value labels
for bar, val in zip(bars, bench_df['mean_ms']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
             f'{val:.1f}ms', ha='center', va='bottom', fontsize=10)

# FPS comparison
ax2 = axes[1]
bars = ax2.bar(bench_df['model'], bench_df['fps'], color=colors, edgecolor='black')
ax2.set_xlabel('Model')
ax2.set_ylabel('Throughput (FPS)')
ax2.set_title('Inference Throughput')
ax2.tick_params(axis='x', rotation=45)

# Add value labels
for bar, val in zip(bars, bench_df['fps']):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, 
             f'{val:.0f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('model_latency_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nLatency Summary:")
print(bench_df[['model', 'mean_ms', 'p95_ms', 'fps']].to_string(index=False))

## 5. Latency Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# Box plot of latency distributions
latency_data = [r['latencies'] for r in benchmark_results]
positions = range(len(MODELS))

bp = ax.boxplot(latency_data, positions=positions, patch_artist=True)

colors = sns.color_palette('husl', len(MODELS))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_xticklabels(list(MODELS.keys()), rotation=45, ha='right')
ax.set_xlabel('Model')
ax.set_ylabel('Latency (ms)')
ax.set_title('Latency Distribution by Model')

# Add median labels
for i, r in enumerate(benchmark_results):
    median = r['p50_ms']
    ax.annotate(f'{median:.1f}', xy=(i, median), xytext=(5, 5),
                textcoords='offset points', fontsize=9)

plt.tight_layout()
plt.savefig('model_latency_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Accuracy Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(MODELS))
width = 0.25

bars1 = ax.bar(x - width, acc_df['mAP@0.5'], width, label='mAP@0.5', color='steelblue')
bars2 = ax.bar(x, acc_df['mAP@0.5:0.95'], width, label='mAP@0.5:0.95', color='coral')
bars3 = ax.bar(x + width, acc_df['precision'], width, label='Precision', color='seagreen')

ax.set_xlabel('Model')
ax.set_ylabel('Score')
ax.set_title('Model Accuracy Metrics')
ax.set_xticks(x)
ax.set_xticklabels(acc_df['model'], rotation=45, ha='right')
ax.legend()
ax.set_ylim(0, 1.0)

# Add grid
ax.yaxis.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('model_accuracy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nAccuracy Summary:")
print(acc_df.to_string(index=False))

## 7. Accuracy vs Latency Trade-off

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

# Merge accuracy and latency data
merged_df = pd.merge(bench_df, acc_df, on='model')

# Scatter plot
colors = sns.color_palette('husl', len(MODELS))
scatter = ax.scatter(merged_df['mean_ms'], merged_df['mAP@0.5'], 
                     s=200, c=colors, edgecolor='black', linewidth=2)

# Add labels
for i, row in merged_df.iterrows():
    ax.annotate(row['model'], (row['mean_ms'], row['mAP@0.5']),
                xytext=(10, 10), textcoords='offset points',
                fontsize=10, fontweight='bold')

ax.set_xlabel('Latency (ms)', fontsize=12)
ax.set_ylabel('mAP@0.5', fontsize=12)
ax.set_title('Accuracy vs Latency Trade-off', fontsize=14)

# Add quadrant annotations
ax.axhline(y=0.75, color='gray', linestyle='--', alpha=0.5)
ax.axvline(x=10, color='gray', linestyle='--', alpha=0.5)

ax.text(5, 0.85, 'Optimal\n(Fast & Accurate)', ha='center', fontsize=10, color='green')
ax.text(18, 0.85, 'Accurate\n(But Slow)', ha='center', fontsize=10, color='orange')
ax.text(5, 0.65, 'Fast\n(But Less Accurate)', ha='center', fontsize=10, color='orange')

plt.tight_layout()
plt.savefig('accuracy_vs_latency.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Precision Comparison (FP32 vs FP16 vs INT8)

In [ ]:
# Filter YOLOv8n variants
precision_models = ['YOLOv8n-FP32', 'YOLOv8n-FP16', 'YOLOv8n-INT8']
precision_bench = bench_df[bench_df['model'].isin(precision_models)].copy()
precision_acc = acc_df[acc_df['model'].isin(precision_models)].copy()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Latency comparison
ax1 = axes[0]
colors = ['#3498db', '#2ecc71', '#e74c3c']
ax1.bar(precision_bench['model'], precision_bench['mean_ms'], color=colors, edgecolor='black')
ax1.set_ylabel('Latency (ms)')
ax1.set_title('Latency by Precision')
ax1.tick_params(axis='x', rotation=45)

# Throughput comparison
ax2 = axes[1]
ax2.bar(precision_bench['model'], precision_bench['fps'], color=colors, edgecolor='black')
ax2.set_ylabel('Throughput (FPS)')
ax2.set_title('Throughput by Precision')
ax2.tick_params(axis='x', rotation=45)

# Accuracy comparison
ax3 = axes[2]
ax3.bar(precision_acc['model'], precision_acc['mAP@0.5'], color=colors, edgecolor='black')
ax3.set_ylabel('mAP@0.5')
ax3.set_title('Accuracy by Precision')
ax3.set_ylim(0.65, 0.80)
ax3.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('precision_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary table
print("\nPrecision Impact Summary:")
print("="*60)
fp32_lat = precision_bench[precision_bench['model'] == 'YOLOv8n-FP32']['mean_ms'].values[0]
for _, row in precision_bench.iterrows():
    speedup = fp32_lat / row['mean_ms']
    acc_row = precision_acc[precision_acc['model'] == row['model']]
    acc = acc_row['mAP@0.5'].values[0]
    print(f"{row['model']:15} | {row['mean_ms']:.1f}ms | {row['fps']:.0f} FPS | "
          f"mAP={acc:.3f} | Speedup: {speedup:.2f}x")

## 9. Recommendations

In [ ]:
print("="*70)
print("MODEL RECOMMENDATIONS")
print("="*70)

print("\n📊 Use Case Recommendations:\n")

recommendations = [
    {
        'use_case': 'Edge Deployment (Jetson Nano)',
        'model': 'YOLOv8n-INT8',
        'reason': 'Lowest latency, fits memory constraints, acceptable accuracy loss',
    },
    {
        'use_case': 'Edge Deployment (Jetson Orin)',
        'model': 'YOLOv8n-FP16',
        'reason': 'Best accuracy/speed balance, no INT8 accuracy loss',
    },
    {
        'use_case': 'Cloud Deployment (High Accuracy)',
        'model': 'YOLOv8m-FP16',
        'reason': 'Highest accuracy, acceptable latency for cloud',
    },
    {
        'use_case': 'Multi-Stream Processing (16+ cameras)',
        'model': 'YOLOv8n-FP16',
        'reason': 'High throughput allows more streams per GPU',
    },
    {
        'use_case': 'Real-time Analytics (<100ms E2E)',
        'model': 'YOLOv8n-INT8',
        'reason': 'Sub-5ms inference leaves headroom for tracking & analytics',
    },
]

for rec in recommendations:
    print(f"🎯 {rec['use_case']}")
    print(f"   Recommended: {rec['model']}")
    print(f"   Reason: {rec['reason']}")
    print()

print("\n" + "="*70)
print("KEY FINDINGS")
print("="*70)
print("""
1. FP16 provides 2x speedup over FP32 with no accuracy loss
2. INT8 provides 2x speedup over FP16 with ~2% mAP loss
3. YOLOv8n-FP16 is the best all-around choice for most deployments
4. Use INT8 only when latency is critical and accuracy loss is acceptable
5. Larger models (YOLOv8s/m) only justified for high-accuracy requirements
""")

## 10. Export Results

In [ ]:
# Export results to JSON
results = {
    'benchmark_date': time.strftime('%Y-%m-%d %H:%M:%S'),
    'configuration': {
        'input_shape': INPUT_SHAPE,
        'batch_size': BATCH_SIZE,
        'iterations': BENCHMARK_ITERATIONS,
    },
    'latency_results': bench_df.drop(columns=['latencies'], errors='ignore').to_dict('records'),
    'accuracy_results': acc_df.to_dict('records'),
}

with open('model_comparison_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("Results exported to model_comparison_results.json")

# Export CSV for easy viewing
merged_df = pd.merge(bench_df[['model', 'mean_ms', 'fps', 'p95_ms']], 
                     acc_df[['model', 'mAP@0.5', 'precision', 'recall']], 
                     on='model')
merged_df.to_csv('model_comparison_summary.csv', index=False)
print("Summary exported to model_comparison_summary.csv")